# seq2seq

## BLEU
* $p_n$是预测中所有n-gram的精度
  * 比如说标签序列`A B C D E F`和预测序列`A B B C D`，有  
    * $p_1 = \frac{4}{5}$  
        因为在预测序列中，当`n-gram = 1`即单个字符时，有`A B C D`四个字符是存在于标签中的  
    * $p_2 = \frac{3}{4}$  
        同理，在预测序列中，`n-gram = 2`时一共有`A B`,`B B`,`B C`,`C D`四组，其中`A B`，`B C`,`C D`在标签序列中是有相同的序列的。
    * $p_3 = \frac{1}{3}$  
        同理可得，不过多赘述
    * $p_4 = 0$

* 因此，我们这样定理BLEU：  
    $$
    exp(min(0,1-\frac{len_{label}}{len_{pred}}))\prod_{n = 1}^{k}p_{n}^{\frac{1}{2^n}}
    $$

In [1]:
import collections
import math
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
class Seq2seqEncoder(d2l.Encoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout = 0, **kwargs):
        super(Seq2seqEncoder).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers, dropout=dropout)

    def forward(self, X, *args):
        X = self.embedding(X)
        X = X.permute(1, 0, 2)
        output, state = self.rnn(X)
        return output, state